# 🧠 Socratic Watchdog — Manual Feature Test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xamzar/socratic-watchdog/blob/main/demo_manual_test.ipynb)

Run every cell top-to-bottom. Each feature has an **Expected** note telling you exactly what you should see, so you can confirm it works.

## Before you start — three prerequisites

1. **On Colab, run section 0a first** to install the package. Locally, save this notebook (Ctrl/Cmd-S) — the `#Test cases` / auto-detect features locate your cell in the *saved* `.ipynb` on disk. Works best in Colab, JupyterLab, or DIVE.
2. **LLM API key (for guiding questions).** Set `SOCRATIC_LLM_API_KEY` (or `DEEPSEEK_API_KEY` / `OPENAI_API_KEY`) in your environment or a `.env`. **Without a key**, the deterministic parts (test fast-path, confetti, honest-failure, all toggles) still work — but instead of a spoken *question* you'll see an amber **“Socrates is offline”** box. That's expected, not a bug.
3. **Audio (optional).** TTS uses the `espeak` backend by default. If `espeak-ng` isn't installed you simply get no sound — the on-screen subtitle boxes still appear. `apt install espeak-ng` (Linux) / `brew install espeak-ng` (macOS) to hear it.

> Reminder: `%%socratic` **must be the first line** of its cell — no comment or code above it.

## 0a. Install (Colab / fresh environment only)
**Skip this if the package is already installed locally.** On Google Colab it isn't — run the next cell first. It installs 0.4.0 straight from GitHub (PyPI still has an older release). **Expected:** pip output ending in `Successfully installed socratic-watchdog-0.4.0`. After it finishes, continue to *0b. Load the extension* below.

In [ ]:
!pip install -q "git+https://github.com/xamzar/socratic-watchdog.git@main"

## 0b. Load the extension
**Expected:** a “🧠 Socratic Watchdog loaded!” banner listing the commands.

In [ ]:
%load_ext socratic_watchdog

## 1. Help
**Expected:** a command reference listing `%%socratic`, `%socratic_task`, `%socratic_audio`, `%socratic_model`, `%socratic_style`, and the `#Test cases` convention.

In [ ]:
%socratic_help

## 2. Fast path — passing hidden tests → silence + confetti 🎉
The `#Test cases` cell below works in **track (auto) mode**: turn on `%socratic_task auto`, write your task in a **markdown** cell, put the `%%socratic` code right under it, and the `#Test cases` cell directly below that. *(An explicit `%socratic_task` skips the cell-below scan — that path uses cached/LLM-generated tests instead.)*

**Expected:** the `%%socratic` cell prints `🧠 Task: …` (echoing the markdown), then — because the code passes the tests below — a green **“Socrates says…”** praise box **and a burst of confetti**, no question. The LLM is skipped entirely.

In [ ]:
%socratic_task auto

**Task:** Write a function `double(n)` that returns n times 2.

In [ ]:
%%socratic
def double(n):
    return n * 2

In [ ]:
#Test cases
assert double(2) == 4
assert double(5) == 10

## 3. Honest failure — wrong code never gets praised
Same track-mode setup, but the code is wrong.

**Expected:**
- **With** an API key: a spoken/subtitled **guiding question** (e.g. *“What does your function return for n=2?”*) — never the fix.
- **Without** a key: a red **“Not passing yet”** box. Either way: **no confetti, no false praise.**

**Task:** Write a function `triple(n)` that returns n times 3.

In [ ]:
%%socratic
def triple(n):
    return n * 2  # wrong on purpose

In [ ]:
#Test cases
assert triple(2) == 6
assert triple(5) == 15

## 4. Audio toggle — `%socratic_audio off` / `on`
**Expected:** `off` prints `🔇 Audio OFF`; after it, cells show the subtitle boxes **but play no sound**. Turning it back `on` restores audio. (Re-run cell 3's wrong-code cell after each toggle to hear the difference.)

In [ ]:
%socratic_audio off

In [ ]:
%socratic_audio on

## 5. Model switching — `%socratic_model`
**Expected:** with no argument, a numbered list of providers with `← current` on the active one. `%socratic_model 1` selects DeepSeek Chat; a custom name is kept verbatim. (Switching model doesn't call the LLM — it just sets which one the next `%%socratic` uses.)

In [ ]:
%socratic_model

In [ ]:
%socratic_model 1

## 6. Style — `%socratic_style brief` / `verbose`
**Expected:** prints `🏛️ Style: brief` / `verbose`. In **brief** mode Socrates' questions are terse and direct; **verbose** is the playful mentor tone (default). Re-run a wrong-code cell after switching to feel the difference (needs API key).

In [ ]:
%socratic_style brief

In [ ]:
%socratic_style verbose

## 7. Debug mode — `%socratic_debug on`
**Expected:** prints `🐛 Debug ON`. After this, running a `%%socratic` cell prints an extra table: test pass/fail counts, which LLM model + how long, TTS backend + time, and end-to-end timing.

In [ ]:
%socratic_debug on

**Task:** Write a function `halve(n)` that returns n divided by 2.

In [ ]:
%%socratic
def halve(n):
    return n / 2

In [ ]:
#Test cases
assert halve(4) == 2

## 8. Timing stats — `%socratic_stats`
**Expected:** a per-step wall-clock table (run_cell, build_prompt, llm_call, parse, tts+display, end-to-end) from the **last** analysis. If you haven't run a `%%socratic` cell yet this session, it says so instead.

In [ ]:
%socratic_stats

In [ ]:
%socratic_debug off

## 9. Exploration mode — `%socratic_explore on`
**Expected:** prints `🔍 Exploration mode ON`. With no task set, Socrates encourages free experimentation instead of nagging you to declare a goal. Turn `off` to return to task-driven mode.

In [ ]:
%socratic_explore on

In [ ]:
%socratic_explore off

## 10. Auto-generated tests — `%socratic_generate_tests` (needs API key)
Set an **explicit** task, then generate hidden tests from it (this is the explicit-task path — LLM writes the tests, no cell-below needed). **Expected:** `🧠 These tests will run silently…`. The tests are cached on disk (keyed by task) so re-running is instant. **Without a key:** `No tests generated.`

In [ ]:
%socratic_task Write a function is_even(n) that returns True for even numbers
%socratic_generate_tests

## 11. Inspect + clear the tests cache
**Expected:** `%socratic_cache` lists cached tasks (hash, test count, age, task preview). `%socratic_cache show` prints the actual asserts for the current task. `%socratic_clear_cache` wipes them and reports how many it cleared.

In [ ]:
%socratic_cache

In [ ]:
%socratic_cache show

In [ ]:
%socratic_clear_cache

## 12. Auto-watch every cell — `%socratic_watch on`
**Expected:** after `on`, you **don't** need `%%socratic` — every normal code cell is analyzed automatically (with a 3s debounce). First set an explicit task so Socrates knows the goal, then run the wrong-code cell and watch it react on its own. `%socratic_off` (or `%socratic_watch off`) stops it.

In [ ]:
%socratic_task Write a function square(n) that returns n squared
%socratic_watch on

In [ ]:
def square(n):
    return n + n  # wrong: adds instead of squares

In [ ]:
%socratic_off

## 13. Reset
**Expected:** `🧠 Context reset.` — clears task, tests, and cached notebook data. You're back to a clean slate.

---
✅ **If every cell matched its Expected note, all documented features work.** Anything that didn't behave as described is worth reporting before publishing to PyPI.

In [ ]:
%socratic_reset